# Análisis de resultados — Gemini

Este notebook analiza los resultados generados por `analisis_texto.py` y `analisis_imagenes.py`.

> **Nota:** si los JSON indican `"modo": "demo"`, las métricas corresponden a la comprobación local del flujo. El proyecto de Google AI Studio utilizado durante la práctica estaba marcado como **Restringido** y devolvía `403 PERMISSION_DENIED` en las llamadas reales.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()

with open(BASE_DIR / "resultados_texto.json", encoding="utf-8") as f:
    texto = json.load(f)

with open(BASE_DIR / "resultados_imagenes.json", encoding="utf-8") as f:
    imagenes = json.load(f)

print("Modo texto:", texto["modo"])
print("Modo imágenes:", imagenes["modo"])
print("Modelo:", texto["modelo"])


## 1. Resultados de texto

In [ ]:
filas_texto = []

for item in texto["resultados"]:
    esperado = item["esperado"]
    obtenido = item["resultado"]

    filas_texto.append({
        "ID": item["id"],
        "Tipo": item["tipo"],
        "Idioma esperado": esperado["idioma"],
        "Idioma Gemini": obtenido["idioma"],
        "Idioma correcto": esperado["idioma"] == obtenido["idioma"],
        "Sentimiento esperado": esperado["sentimiento"],
        "Sentimiento Gemini": obtenido["sentimiento"],
        "Sentimiento correcto": esperado["sentimiento"] == obtenido["sentimiento"],
        "Resumen": obtenido["resumen"],
    })

df_texto = pd.DataFrame(filas_texto)
df_texto


In [ ]:
metricas_texto = pd.DataFrame([
    {"Tarea": "Detección de idioma", "Precisión": texto["metricas"]["idioma"]},
    {"Tarea": "Sentimiento", "Precisión": texto["metricas"]["sentimiento"]},
    {"Tarea": "Entidades", "Precisión": texto["metricas"]["entidades_recall"]},
    {"Tarea": "Media global", "Precisión": texto["metricas"]["media_global"]},
])

metricas_texto["Porcentaje"] = (metricas_texto["Precisión"] * 100).round(1)
metricas_texto


## 2. Resultados de visión

In [ ]:
filas_imagenes = []

for item in imagenes["resultados"]:
    p = item["precision"]
    filas_imagenes.append({
        "Imagen": item["archivo"],
        "Tipo": item["tipo"],
        "Tarea principal": item["tarea_principal"],
        "OCR %": round(p["ocr"] * 100, 1),
        "Descripción %": round(p["descripcion"] * 100, 1),
        "Escena %": round(p["escena"] * 100, 1),
        "Objetos %": round(p["objetos"] * 100, 1),
    })

df_imagenes = pd.DataFrame(filas_imagenes)
df_imagenes


In [ ]:
metricas_imagenes = pd.DataFrame([
    {"Tarea": "OCR", "Precisión": imagenes["metricas"]["ocr"]},
    {"Tarea": "Descripción", "Precisión": imagenes["metricas"]["descripcion"]},
    {"Tarea": "Clasificación de escena", "Precisión": imagenes["metricas"]["escena"]},
    {"Tarea": "Detección de objetos", "Precisión": imagenes["metricas"]["objetos"]},
    {"Tarea": "Media global", "Precisión": imagenes["metricas"]["media_global"]},
])

metricas_imagenes["Porcentaje"] = (metricas_imagenes["Precisión"] * 100).round(1)
metricas_imagenes


## 3. Comparación visual

In [ ]:
comparativa = pd.concat([
    metricas_texto[metricas_texto["Tarea"] != "Media global"].assign(Bloque="Texto"),
    metricas_imagenes[metricas_imagenes["Tarea"] != "Media global"].assign(Bloque="Visión"),
], ignore_index=True)

ax = comparativa.plot(
    kind="bar",
    x="Tarea",
    y="Porcentaje",
    legend=False,
    figsize=(10, 5),
)
ax.set_ylabel("Precisión (%)")
ax.set_ylim(0, 105)
ax.set_title("Comparación de precisión por tarea")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 4. Conclusión

La práctica amplía el ejercicio anterior incorporando análisis avanzado de texto y visión.

- En **texto** se evalúan detección de idioma, entidades, sentimiento y resumen.
- En **visión** se evalúan OCR, descripción, clasificación de escena y objetos.
- Los scripts generan resultados estructurados en JSON y calculan métricas comparables.
- El notebook centraliza el análisis y permite detectar qué tareas funcionan mejor.

Mientras el proyecto de Google AI Studio esté restringido, el modo demo permite verificar la estructura del programa, la generación de archivos y el análisis. Cuando el acceso real quede habilitado, basta con ejecutar los scripts sin `--demo` para sustituir los resultados simulados por respuestas reales de Gemini.
